In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict 
import os

try:
    import dataframe_image as dfi
except:
    pass

import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14,'figure.dpi':300})


idx = pd.IndexSlice # Helper for multi-index lookups

In [34]:
import pandas as pd

results_dir = r'C:/Users/tom.walsh/OneDrive - Burnet Institute/Desktop/Tom Walsh PHD work/PhD work/Papers/Paper 1/results/'

regions = ['SA', 'SSA', 'Other']

dfs = []

for r in regions:
    path = f'epi_df_{r}90.xlsx'
    df = pd.read_excel(results_dir + path)
    df = df.set_index(['Scenario','Population Name','Year','Cause','Measure'])
    dfs.append(df)

# Sum across regions, aligned on the exact same index (including Year)
epi_df = dfs[0].copy()

for df in dfs[1:]:
    epi_df = epi_df.add(df, fill_value=0)

# Optional: make sure you keep the same column order / dtypes
epi_df = epi_df.sort_index()

In [42]:
num_births = epi_df['Value'].unstack()['births'].dropna().groupby(level='Scenario').sum()
num_births

Scenario
ai_ultrasound               3.894173e+09
non_invasive_hb_test        3.894173e+09
pe_screening                3.894173e+09
soc                         3.894173e+09
soc_diagnostics_scale_up    3.894173e+09
soc_hb_test                 3.894173e+09
soc_iron_tablets            3.894173e+09
soc_pe_treatment            3.894173e+09
soc_pe_urine_dipstick       3.894173e+09
soc_rutf                    3.894173e+09
soc_scale_up                3.894173e+09
soc_ultrasound              3.894173e+09
Name: births, dtype: float64

In [36]:
df = epi_df['Value'].unstack()['cases'].unstack().groupby(level='Scenario').sum()
df = df.drop('demographics',axis=1)

approx_incidence = df.divide(num_births, axis=0)
approx_incidence['hemorrhage']

Scenario
ai_ultrasound               0.102544
non_invasive_hb_test        0.102472
pe_screening                0.102544
soc                         0.102544
soc_diagnostics_scale_up    0.101271
soc_hb_test                 0.102531
soc_iron_tablets            0.101820
soc_pe_treatment            0.102544
soc_pe_urine_dipstick       0.102544
soc_rutf                    0.102544
soc_scale_up                0.101726
soc_ultrasound              0.102544
Name: hemorrhage, dtype: float64

In [37]:
df = approx_incidence/approx_incidence.loc['soc']
s = df.style.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#4287f5')  
s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ff0000')  

Cause,anemia,anemia_pp,hemorrhage,maternal sepsis,neonatal encephalopathy,neonatal sepsis,non-rds preterm,obstructed labour,other neonatal conditions,preeclampsia,preterm,preterm & sga,rds,sga,stillbirth,stunting,wasting,wasting 19 months to 59 months,wasting 6 months to 18 months,wasting 8d to 5 months
Scenario,,,,,,,,,,,,,,,,,,,,
ai_ultrasound,1.000000,1.000055,1.000000,1.000000,0.919771,1.000635,nan,1.000000,nan,1.000000,1.000034,1.000044,1.000769,1.000051,nan,1.000669,nan,nan,1.001934,1.001851
non_invasive_hb_test,0.995507,0.997230,0.999297,1.000000,1.000007,1.000001,nan,1.000000,nan,1.000000,1.000000,0.998666,1.000002,0.998529,nan,1.000002,nan,nan,1.000003,0.999591
pe_screening,1.000000,1.000000,1.000000,1.000000,1.000468,1.000328,nan,1.000000,nan,0.943471,0.986807,0.985229,0.986904,0.998546,nan,1.000343,nan,nan,1.000515,1.000135
soc,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,nan,1.000000,nan,1.000000,1.000000,1.000000,1.000000,1.000000,nan,1.000000,nan,nan,1.000000,1.000000
soc_diagnostics_scale_up,0.919548,0.950283,0.987590,1.000000,0.920405,1.001056,nan,1.000000,nan,0.929339,0.982457,0.957651,0.983284,0.973288,nan,1.001110,nan,nan,1.002605,0.995127
soc_hb_test,0.999262,0.999545,0.999881,1.000000,1.000001,1.000000,nan,1.000000,nan,1.000000,1.000000,0.999786,1.000000,0.999761,nan,1.000000,nan,nan,1.000000,0.999935
soc_iron_tablets,0.954102,0.971630,0.992946,1.000000,1.000068,1.000014,nan,1.000000,nan,1.000000,1.000003,0.986633,1.000018,0.985800,nan,1.000015,nan,nan,1.000023,0.996097
soc_pe_treatment,1.000000,1.000000,1.000000,1.000000,1.000003,1.000002,nan,1.000000,nan,1.000000,0.999772,0.999767,0.999768,0.999997,nan,1.000002,nan,nan,1.000005,1.000004
soc_pe_urine_dipstick,1.000000,1.000000,1.000000,1.000000,1.000004,1.000002,nan,1.000000,nan,1.000000,0.999740,0.999738,0.999735,0.999997,nan,1.000002,nan,nan,1.000005,1.000004


In [39]:
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
df = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
# df = df.drop([ 'B000000001010101111111111', 'B000000001100101111111111', 'B000000001110101111111111', 'B000011001000001111111111',
#               'B000000111000001111111111', 'B000011111001001111111111', 'B000000111001001111111111', 'B000011111000001111111111', 'B000011001001001111111111'])
#df.index = df.index.map(branch_name_product)
df = df.drop('demographics',axis=1)

df

df = df-df.loc['soc']
df = df.dropna(how='all', axis=1)
df = df.sort_index(axis=1)
df=df.T

s = df.style
s = s.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#ccdcfc') 
s = s.highlight_between(left=0.0, right=0.99, inclusive='neither', color='#4287f5') 
s = s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ffdddd')  
s = s.highlight_between(left=1.01, right=np.inf, inclusive='neither', color='#ff0000')  
# s = s.format(lambda x: f'{(x)*1:+.2g}%' if not np.isnan(x) else '-')
s.set_table_styles([
    {"selector": "td,th", "props": "height: 3em;"},
    {"selector": "th", "props": [("border", "1px solid")]}   
])
s = s.set_properties(**{'text-align': 'center','border':'1px solid black'})

try:
    dfi.export(s,str(fig_dir/'daly_impact.png'))
except:
    pass


s

Scenario,ai_ultrasound,non_invasive_hb_test,pe_screening,soc,soc_diagnostics_scale_up,soc_hb_test,soc_iron_tablets,soc_pe_treatment,soc_pe_urine_dipstick,soc_rutf,soc_scale_up,soc_ultrasound
Cause,,,,,,,,,,,,
anemia,0.000000,-193709.924950,0.000000,0.000000,-3166946.580184,-32490.283966,-1812055.108623,0.000000,0.000000,0.000000,-2048596.937830,0.000000
anemia_pp,19321.892571,-439935.642711,-8.641198,0.000000,-7088722.226836,-73085.516935,-4042650.596877,-0.131412,-0.147302,0.000000,-4569877.329480,7960.791402
hemorrhage,-4102956.387645,-47678.899949,1996.697812,0.000000,-4717403.187446,-8054.601529,-416210.429499,32.049563,36.513639,0.000000,-2123667.212687,-1681753.043463
maternal sepsis,544.371124,6.717016,-0.218111,0.000000,633.517112,1.237265,54.821931,-0.003655,-0.004260,0.000000,284.185788,223.393531
neonatal encephalopathy,-91742209.650604,10657.137092,347993.153021,0.000000,-91120572.932594,1813.032850,89734.946311,3570.780225,4266.245417,0.000000,-38210633.017384,-38361626.065956
neonatal sepsis,653282.122384,636.034267,141666.945924,0.000000,875415.868118,111.532251,5335.438644,1345.778828,1615.662090,0.000000,276414.059092,267008.767611
non-rds preterm,-15759962.294600,-60808.215448,-4382807.253989,0.000000,-23648224.300776,-9957.029942,-547091.744776,-96381.493806,-112458.332735,0.000000,-7033603.275306,-6171603.404641
obstructed labour,-1622335.510402,142.223922,400.983245,0.000000,-1619396.571077,24.351006,1228.650009,7.325848,8.723584,0.000000,-665696.608222,-667812.466015
other neonatal conditions,524694.934478,592.927585,116396.136108,0.000000,709042.571546,99.101838,5177.067366,1044.343560,1228.047005,0.000000,222878.506320,214547.482170


In [30]:
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
df = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum()

In [23]:
df = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
# df = df.drop([ 'B000000001010101111111111', 'B000000001100101111111111', 'B000000001110101111111111', 'B000011001000001111111111',
#               'B000000111000001111111111', 'B000011111001001111111111', 'B000000111001001111111111', 'B000011111000001111111111', 'B000011001001001111111111'])
#df.index = df.index.map(branch_name_product)
df = df.drop('demographics',axis=1)

df

df = df/df.loc['soc']
df = df.dropna(how='all', axis=1)
df = df.sort_index(axis=1)
df = df.T

s = df.style
s = s.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#ccdcfc') 
s = s.highlight_between(left=0.0, right=0.99, inclusive='neither', color='#4287f5') 
s = s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ffdddd')  
s = s.highlight_between(left=1.01, right=np.inf, inclusive='neither', color='#ff0000')  
s = s.format(lambda x: f'{(x-1)*100:+.2g}%' if not np.isnan(x) else '-')
s.set_table_styles([
    {"selector": "td,th", "props": "height: 3em;"},
    {"selector": "th", "props": [("border", "1px solid")]}   
])
s = s.set_properties(**{'text-align': 'center','border':'1px solid black'})

try:
    dfi.export(s,str(fig_dir/'daly_impact.png'))
except:
    pass

out = r'C:/Users/tom.walsh/OneDrive - Burnet Institute/Desktop/Tom Walsh PHD work/PhD work/Papers/Paper 1/results/daly_impact.xlsx'

with pd.ExcelWriter(out, engine="openpyxl") as writer:
    df.to_excel(writer)
s

Scenario,ai_ultrasound,non_invasive_hb_test,pe_screening,soc,soc_diagnostics_scale_up,soc_hb_test,soc_iron_tablets,soc_pe_treatment,soc_pe_urine_dipstick,soc_rutf,soc_scale_up,soc_ultrasound
Cause,,,,,,,,,,,,
anemia,+0%,-0.59%,+0%,+0%,-7.3%,+0%,-4.2%,+0%,+0%,+0%,-4.2%,+0%
anemia_pp,+0.0085%,-0.31%,-7e-06%,+0%,-3.8%,+0%,-2.2%,+0%,-1.5e-09%,+0%,-2.2%,+0.0065%
hemorrhage,-4.3%,-0.08%,+0.0039%,+0%,-5.1%,+0%,-0.54%,+0%,+2.5e-06%,+0%,-3.8%,-3.3%
maternal sepsis,+0.0016%,+3.2e-05%,-1.2e-06%,+0%,+0.0019%,+0%,+0.00019%,+0%,-1.3e-09%,+0%,+0.0014%,+0.0012%
neonatal encephalopathy,-6.6%,+0.0012%,+0.046%,+0%,-6.5%,+0%,+0.0081%,+0%,+4.7e-05%,+0%,-5%,-5%
neonatal sepsis,+0.11%,+0.00017%,+0.043%,+0%,+0.15%,+0%,+0.0011%,+0%,+4.6e-05%,+0%,+0.083%,+0.082%
non-rds preterm,-2.1%,-0.014%,-1.1%,+0%,-3.4%,+0%,-0.096%,+0%,-0.002%,+0%,-1.7%,-1.6%
obstructed labour,-6%,+0.00085%,+0.0027%,+0%,-5.9%,+0%,+0.0058%,+0%,+5.2e-06%,+0%,-4.6%,-4.6%
other neonatal conditions,+0.083%,+0.00015%,+0.034%,+0%,+0.12%,+0%,+0.001%,+0%,+2.7e-05%,+0%,+0.064%,+0.063%


In [24]:
df = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
# df = df.drop([ 'B000000001010101111111111', 'B000000001100101111111111', 'B000000001110101111111111', 'B000011001000001111111111',
#               'B000000111000001111111111', 'B000011111001001111111111', 'B000000111001001111111111', 'B000011111000001111111111', 'B000011001001001111111111'])
#df.index = df.index.map(branch_name_product)
df = df.drop('demographics',axis=1)

df

df = df.loc['soc']
df

Cause
anemia                            2.872918e+07
anemia_pp                         1.230559e+08
hemorrhage                        5.171044e+07
maternal sepsis                   1.826393e+07
neonatal encephalopathy           7.640525e+08
neonatal sepsis                   3.264764e+08
non-rds preterm                   3.916371e+08
obstructed labour                 1.462137e+07
other neonatal conditions         3.394180e+08
preeclampsia                      3.024640e+07
preterm                           0.000000e+00
preterm & sga                     0.000000e+00
rds                               5.431101e+08
sga                               2.892757e+08
stillbirth                        1.905200e+09
stunting                          9.037197e+07
wasting                           1.027892e+09
wasting 19 months to 59 months    0.000000e+00
wasting 6 months to 18 months     0.000000e+00
wasting 8d to 5 months            0.000000e+00
Name: soc, dtype: float64

In [10]:
##OUTPUTS SECTION##
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
dalys = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
dalys_averted = dalys.loc['soc'] - dalys
dalys_averted

Cause,anemia,anemia_pp,demographics,hemorrhage,maternal sepsis,neonatal encephalopathy,neonatal sepsis,non-rds preterm,obstructed labour,other neonatal conditions,...,preterm,preterm & sga,rds,sga,stillbirth,stunting,wasting,wasting 19 months to 59 months,wasting 6 months to 18 months,wasting 8d to 5 months
Scenario,,,,,,,,,,,,,,,,,,,,,
ai_ultrasound,0.000000e+00,-1.637173e+04,0.0,2.941391e+06,-285.485596,3.008875e+07,-164610.361740,5.922165e+06,470602.649252,-184536.441961,...,0.0,0.0,1.564201e+07,4.290313e+06,3.120827e+07,-51592.120840,-5.707181e+05,0.0,0.0,0.0
non_invasive_hb_test,9.228831e+04,2.133067e+05,0.0,2.794218e+04,-2.453189,-7.227978e+03,-348.537713,4.060238e+04,-93.810264,-394.118740,...,0.0,0.0,-1.156252e+03,3.540499e+05,-9.010899e+03,15336.989538,1.824838e+05,0.0,0.0,0.0
pe_screening,0.000000e+00,7.624834e+00,0.0,-1.404746e+03,0.129004,-1.184110e+05,-33646.111653,1.386636e+06,-115.406730,-37899.206339,...,0.0,0.0,1.904917e+06,6.908287e+04,1.600677e+07,23406.478412,-2.506528e+04,0.0,0.0,0.0
soc,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.0,0.0
soc_diagnostics_scale_up,1.415935e+06,3.227561e+06,0.0,3.261901e+06,-313.757858,2.982540e+07,-220301.974417,8.695708e+06,468981.996083,-247360.574969,...,0.0,0.0,1.855332e+07,9.732084e+06,5.520403e+07,227001.355254,2.272636e+06,0.0,0.0,0.0
soc_hb_test,1.467215e+04,3.350035e+04,0.0,4.353109e+03,-0.382223,-1.132756e+03,-54.497377,6.355474e+03,-14.835084,-61.626438,...,0.0,0.0,-1.795283e+02,5.552359e+04,-1.413855e+03,2407.079196,2.865653e+04,0.0,0.0,0.0
soc_iron_tablets,8.103189e+05,1.848518e+06,0.0,2.429423e+05,-21.332172,-6.291047e+04,-3048.707584,3.557620e+05,-830.211870,-3447.806456,...,0.0,0.0,-9.994652e+03,3.110256e+06,-7.888230e+04,134554.296496,1.606198e+06,0.0,0.0,0.0
soc_pe_treatment,0.000000e+00,1.112218e-01,0.0,-2.028865e+01,0.001864,-1.034663e+03,-273.712576,2.946288e+04,-1.652364,-307.683950,...,0.0,0.0,4.222067e+04,-2.112708e+03,8.963368e+04,479.808089,-1.166056e+03,0.0,0.0,0.0
soc_pe_urine_dipstick,0.000000e+00,1.242247e-01,0.0,-2.264991e+01,0.002081,-1.155212e+03,-305.264826,3.287428e+04,-1.843783,-343.211074,...,0.0,0.0,4.711550e+04,-2.357050e+03,9.995454e+04,535.191519,-1.301892e+03,0.0,0.0,0.0


In [16]:
regions = ['Other', 'SA','SSA']
scenarios = ['90']
shape = np.shape(epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm']))

    #work out how to separate indirect and direct
for scenario in scenarios:
    soc_indirect = []
    dalys_averted_sum = []
    outputs_1 = pd.DataFrame()
    outputs_2 = pd.DataFrame()
    outputs_3 = pd.DataFrame()
    outputs_4 = pd.DataFrame()
    dalys_averted_out = pd.DataFrame()
    soc_out = pd.DataFrame()
    soc_scale_up_out = pd.DataFrame()
    soc_diagnostics_scale_up_out = pd.DataFrame()
    soc_dalys = np.zeros(shape)
    soc_cases = np.zeros(shape)
    soc_deaths = np.zeros(shape)
    soc_scale_up_dalys = np.zeros(shape)
    soc_scale_up_cases = np.zeros(shape)
    soc_scale_up_deaths = np.zeros(shape)
    soc_diagnostics_scale_up_dalys = np.zeros(shape)
    soc_diagnostics_scale_up_cases = np.zeros(shape)
    soc_diagnostics_scale_up_deaths = np.zeros(shape)
    for region in regions:
        path = f'epi_df_{region}_{scenario}.xlsx'
        epi_df = pd.read_excel(results_dir / path)
        epi_df = epi_df.set_index(['Scenario', 'Population Name', 'Year', 'Cause', 'Measure'])
        epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
        dalys = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
        dalys_averted = dalys.loc['soc'] - dalys

        # Calculate total DALYs


        # Calculate direct impacts
        direct_impacts_data = {
            region: [
                dalys_averted.loc['soc_rutf', ['wasting', 'wasting 19 months to 59 months', 'wasting 6 months to 18 months', 'wasting 8d to 5 months']].sum(axis=0),
                dalys_averted.loc['soc_iron_tablets', ['anemia']].sum(axis=0),
                dalys_averted.loc['soc_hb_test', ['anemia']].sum(axis=0),
                dalys_averted.loc['soc_pe_urine_dipstick', ['preeclampsia']].sum(axis=0),
                dalys_averted.loc['soc_ultrasound', ['obstructed labour', 'hemorrhage', 'preterm', 'non-rds preterm']].sum(axis=0),
                dalys_averted.loc['soc_pe_treatment', []].sum(axis=0),
                dalys_averted.loc['non_invasive_hb_test', ['anemia']].sum(axis=0),
                dalys_averted.loc['pe_screening', ['preeclampsia']].sum(axis=0),
                dalys_averted.loc['ai_ultrasound', ['obstructed labour', 'hemorrhage', 'preterm', 'non-rds preterm']].sum(axis=0)
            ]
        }
        direct_impacts = pd.DataFrame(direct_impacts_data)

        # Calculate DALYs averted sum
        dalys_averted_sum_data = {
            region: [
                dalys_averted.loc['soc_rutf'].sum(axis=0),
                dalys_averted.loc['soc_iron_tablets'].sum(axis=0),
                dalys_averted.loc['soc_hb_test'].sum(axis=0),
                dalys_averted.loc['soc_pe_urine_dipstick'].sum(axis=0),
                dalys_averted.loc['soc_ultrasound'].sum(axis=0),
                dalys_averted.loc['soc_pe_treatment'].sum(axis=0),
                dalys_averted.loc['non_invasive_hb_test'].sum(axis=0),
                dalys_averted.loc['pe_screening'].sum(axis=0),
                dalys_averted.loc['ai_ultrasound'].sum(axis=0)
            ]
        }
        dalys_averted_sum = pd.DataFrame(dalys_averted_sum_data)

        # Calculate indirect impacts
        indirect_impacts = dalys_averted_sum - direct_impacts

        # Adjust index
        direct_impacts.index = direct_impacts.index * 2
        indirect_impacts.index = indirect_impacts.index * 2 + 1

        # Concatenate direct and indirect impacts
        dalys_averted_out_region = pd.concat([direct_impacts, indirect_impacts]).sort_index()

        # Append total DALYs
        soc_all = pd.DataFrame({region: [dalys.loc['soc'].sum()]})
        soc_scale_up_total_averted = pd.DataFrame({region: [dalys_averted.loc['soc_scale_up'].sum()]})
        soc_diagnostics_scale_up_total_averted = pd.DataFrame({region: [dalys_averted.loc['soc_diagnostics_scale_up'].sum()]})
        dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])

        # Concatenate with previous results
        dalys_averted_out = pd.concat([dalys_averted_out, dalys_averted_out_region], axis=1)


        ## Outputs for key scenarios##
        soc_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_deaths_region = epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm']) + epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])


        soc_scale_up_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_scale_up_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_scale_up_deaths_region = (epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+ epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc']).drop(['demographics', 'preterm & sga', 'preterm'])
        #soc_scale_up_region = {'Dalys'+region:soc_scale_up_dalys,
        #            'Cases'+region:soc_scale_up_cases,
        #           'Deaths'+region:soc_scale_up_deaths}
        
        soc_diagnostics_scale_up_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_diagnostics_scale_up_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_diagnostics_scale_up_deaths_region = (epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+ epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc']).drop(['demographics', 'preterm & sga', 'preterm'])
        # soc_diagnostics_scale_up_region = {'Dalys'+region:soc_scale_up_dalys,
        #            'Cases'+region:soc_diagnostics_scale_up_cases,
        #           'Deaths'+region:soc_diagnostics_scale_up_deaths}
        
        # #Sum regions together
        soc_dalys = soc_dalys + soc_dalys_region
        soc_cases = soc_cases +  soc_cases_region
        soc_deaths = soc_deaths + soc_deaths_region

        soc_scale_up_dalys = soc_scale_up_dalys + soc_scale_up_dalys_region
        soc_scale_up_cases = soc_scale_up_cases + soc_scale_up_cases_region
        soc_scale_up_deaths = soc_scale_up_deaths +soc_scale_up_deaths_region

        soc_diagnostics_scale_up_dalys = soc_diagnostics_scale_up_dalys + soc_diagnostics_scale_up_dalys_region
        soc_diagnostics_scale_up_cases = soc_diagnostics_scale_up_cases + soc_diagnostics_scale_up_cases_region
        soc_diagnostics_scale_up_deaths = soc_diagnostics_scale_up_deaths + soc_diagnostics_scale_up_deaths_region

        ##Outputs for key outputs table

        maternal_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['maternal deaths'].dropna().groupby(level='Scenario').sum()
        maternal_mortalities_averted_1 = maternal_mortalities["soc"]-maternal_mortalities["soc_scale_up"]
        maternal_mortalities_averted_2 = maternal_mortalities["soc"]-maternal_mortalities["soc_diagnostics_scale_up"]
        child_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['child deaths'].dropna().groupby(level='Scenario').sum()
        child_mortalities_averted_1 = child_mortalities["soc"]-child_mortalities["soc_scale_up"]
        child_mortalities_averted_2 = child_mortalities["soc"]-child_mortalities["soc_diagnostics_scale_up"]
        neonatal_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['neonatal deaths'].dropna().groupby(level='Scenario').sum()
        neonatal_mortalities_averted_1 = neonatal_mortalities["soc"]-neonatal_mortalities["soc_scale_up"]
        neonatal_mortalities_averted_2 = neonatal_mortalities["soc"]-neonatal_mortalities["soc_diagnostics_scale_up"]
        stillbirth_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['stillbirth deaths'].dropna().groupby(level='Scenario').sum()
        stillbirth_mortalities_averted_1 = stillbirth_mortalities["soc"]-stillbirth_mortalities["soc_scale_up"]
        stillbirth_mortalities_averted_2 = stillbirth_mortalities["soc"]-stillbirth_mortalities["soc_diagnostics_scale_up"]
        dalys = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['dalys'].dropna().groupby(level='Scenario').sum()
        dalys_averted_1 = dalys["soc"]-dalys["soc_scale_up"]
        dalys_averted_2 = dalys["soc"]-dalys["soc_diagnostics_scale_up"]

        df_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
        df_region = df_region.drop('demographics',axis=1)
        df_region = df_region.loc['soc'] - df_region
        df_region = df_region.dropna(how='all', axis=1).sort_index(axis=1).T

        outputs_1_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities_averted_1,
            'Child mortalities averted': child_mortalities_averted_1,
            'Neonatal mortalities averted': neonatal_mortalities_averted_1,
            'Stillbirth mortalities averted': stillbirth_mortalities_averted_1,
            'Dalys averted': dalys_averted_1
        }, index = [region])

        outputs_2_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities_averted_2,
            'Child mortalities averted': child_mortalities_averted_2,
            'Neonatal mortalities averted': neonatal_mortalities_averted_2,
            'Stillbirth mortalities averted': stillbirth_mortalities_averted_2,
            'Dalys averted': dalys_averted_2
        }, index = [region])

        outputs_3_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities["soc"],
            'Child mortalities averted': child_mortalities["soc"],
            'Neonatal mortalities averted': neonatal_mortalities["soc"],
            'Stillbirth mortalities averted': stillbirth_mortalities["soc"],
            'Dalys averted': dalys["soc"]
        }, index = [region])
        
        outputs_1 = pd.concat([outputs_1, outputs_1_region])
        outputs_2 = pd.concat([outputs_2, outputs_2_region])
        outputs_3 = pd.concat([outputs_3, outputs_3_region])
        outputs_4 = outputs_4.add(df_region, fill_value=0)

    index = ['soc_total', 'soc_rutf_direct', 'soc_rutf_indirect', 'soc iron tablets direct', 'soc iron tablets indirect', 'SOC Hb test direct', 'SOC Hb test indirect', 'SOC pre-eclampsia UDS direct', 'SOC pre-eclampsia UDS indirect', 
             'SOC ultrasound direct','SOC ultrasound indirect', 'soc preeclampsia treatment direct', 'soc preeclampsia treatment indirect',
            'non invasive hb test direct', 'non invasive hb test indirect', 'preeclampsia screening and treatment direct', 'preeclampsia screening and treatment indirect', 'AI ultrasound direct', 'AI ultrasound indirect', 'SOC scale up', 'SOC + diagnostics scale up']
    dalys_averted_out = dalys_averted_out.set_index([pd.Index(index)])
    soc_out = pd.concat([soc_dalys, soc_cases, soc_deaths], axis = 1)
    soc_scale_up_out = pd.concat([soc_scale_up_dalys, soc_scale_up_cases, soc_scale_up_deaths], axis = 1)
    soc_diagnostics_scale_up_out = pd.concat([soc_diagnostics_scale_up_dalys, soc_diagnostics_scale_up_cases, soc_diagnostics_scale_up_deaths], axis = 1)
    soc_out.columns = ["soc_dalys", "soc_cases", "soc_deaths"]
    soc_scale_up_out.columns = ["dalys", "cases", "deaths"]
    soc_diagnostics_scale_up_out.columns = ["dalys", "cases", "deaths"]
    #Create totals row
    dalys_averted_out["Total"] = dalys_averted_out.sum(axis=1)
    outputs_1.loc["Total"] = outputs_1.sum()
    outputs_2.loc["Total"] = outputs_2.sum()
    outputs_3.loc["Total"] = outputs_3.sum()


C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])
C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])
C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagn

In [17]:
dalys_averted_out_formatted = dalys_averted_out.applymap("{:,.0f}".format)
soc_out_formatted = (soc_out/1000000).applymap("{:,.1f}".format)
soc_scale_up_out_formatted = (soc_scale_up_out/1000000).applymap("{:,.1f}".format)
soc_diagnostics_scale_up_out_formatted = (soc_diagnostics_scale_up_out/1000000).applymap("{:,.1f}".format)
outputs_1_formatted = outputs_1.applymap("{:,.0f}".format)
outputs_2_formatted = outputs_2.applymap("{:,.0f}".format)
outputs_3_formatted = outputs_3.applymap("{:,.0f}".format)
outputs_4_formatted = outputs_4.applymap("{:,.0f}".format)
name = "analysis_90_20250529.csv"
with pd.ExcelWriter(results_dir / name) as writer:
    dalys_averted_out_formatted.to_excel(writer, sheet_name='dalys averted by scenario')
    soc_out_formatted.to_excel(writer, sheet_name = 'SOC by cause')
    soc_scale_up_out_formatted.to_excel(writer, sheet_name = 'SOC SU by cause')
    soc_diagnostics_scale_up_out_formatted.to_excel(writer, sheet_name = 'SOC and diagnostics SU by cause')
    outputs_1_formatted.to_excel(writer, sheet_name='key outputs averted SOC')
    outputs_2_formatted.to_excel(writer, sheet_name='key outputs averted SOC+ Diag')
    outputs_3_formatted.to_excel(writer, sheet_name='baseline')
    outputs_4_formatted.to_excel(writer, sheet_name='daly reductions')